# Cycle 1 — Tuning (Chronological Split)

Same `RandomizedSearchCV` configuration as `notebooks/cycle1_tuning.ipynb`. Only the train/test split is changed to chronological. Cross-validation is still `StratifiedKFold(n_splits=5)` over the **training** portion (which is itself contiguous in time) so the inner CV remains a defensible model-selection procedure on past data.

In [1]:
import sys, os

# Locate project root (folder containing data/, models/, notebooks/)
_here = os.getcwd()
while not os.path.isdir(os.path.join(_here, 'data')):
    _p = os.path.dirname(_here)
    if _p == _here: raise RuntimeError('project root not found')
    _here = _p
if _here not in sys.path:
    sys.path.insert(0, _here)

from config import Paths, ensure_dirs
ensure_dirs()  # creates models/cycle1-3 if missing

## Setup & data

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

# Premier League — chronological by Season
df1 = pd.read_csv(str(Paths.PL_MATCHES_PROCESSED)).sort_values('Season').reset_index(drop=True)
split_idx1 = int(len(df1) * 0.8)

# Drop FTR (target) and Season (metadata used for splitting, not a feature)
X1_train = df1.iloc[:split_idx1].drop(columns=['FTR', 'Season'])
y1_train = df1.iloc[:split_idx1]['FTR']
X1_test  = df1.iloc[split_idx1:].drop(columns=['FTR', 'Season'])
y1_test  = df1.iloc[split_idx1:]['FTR']

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f'Train: {len(X1_train)} matches | Test: {len(X1_test)} matches')
print(f'Features ({X1_train.shape[1]}): {list(X1_train.columns)}')


Train: 5472 matches | Test: 1368 matches
Features (33): ['HomeTeam', 'AwayTeam', 'HTGS', 'ATGS', 'HTGC', 'ATGC', 'HTP', 'ATP', 'HM1', 'HM2', 'HM3', 'HM4', 'HM5', 'AM1', 'AM2', 'AM3', 'AM4', 'AM5', 'MW', 'HTFormPts', 'ATFormPts', 'HTWinStreak3', 'HTWinStreak5', 'HTLossStreak3', 'HTLossStreak5', 'ATWinStreak3', 'ATWinStreak5', 'ATLossStreak3', 'ATLossStreak5', 'HTGD', 'ATGD', 'DiffPts', 'DiffFormPts']


## XGBoost search — Dataset 1

In [3]:
xgb_param_grid = {
    'n_estimators':     [100,200,300,500],
    'max_depth':        [3,4,5,6],
    'learning_rate':    [0.01,0.05,0.1,0.2],
    'subsample':        [0.7,0.8,1.0],
    'colsample_bytree': [0.7,0.8,1.0],
    'min_child_weight': [1,3,5],
    'gamma':            [0,0.1,0.2],
}

xgb1 = XGBClassifier(random_state=42, eval_metric='mlogloss', verbosity=0)
search_d1 = RandomizedSearchCV(xgb1, xgb_param_grid, n_iter=50, cv=cv,
                                scoring='accuracy', random_state=42, n_jobs=-1, verbose=1)
search_d1.fit(X1_train, y1_train)

print('Best params:', search_d1.best_params_)
print(f'Best CV accuracy: {search_d1.best_score_*100:.2f}%')
y_pred_xgb_d1 = search_d1.best_estimator_.predict(X1_test)
print(f'Test accuracy:    {accuracy_score(y1_test, y_pred_xgb_d1)*100:.2f}%')
print()
print(classification_report(y1_test, y_pred_xgb_d1, target_names=['Away Win','Draw','Home Win']))

Fitting 5 folds for each of 50 candidates, totalling 250 fits


Best params: {'subsample': 0.8, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.01, 'gamma': 0.1, 'colsample_bytree': 0.7}
Best CV accuracy: 53.02%
Test accuracy:    52.85%

              precision    recall  f1-score   support

    Away Win       0.54      0.44      0.49       406
        Draw       0.28      0.04      0.07       348
    Home Win       0.54      0.86      0.66       614

    accuracy                           0.53      1368
   macro avg       0.45      0.45      0.41      1368
weighted avg       0.47      0.53      0.46      1368



## Side-by-side comparison: Chronological vs Random tuning

The random-split tuning notebook (`../cycle1_tuning.ipynb`) reports XGBoost Tuned on Dataset 1 at **52.78%**. Chronological tuning gives the deployment-honest number — typically slightly lower because the test set genuinely comes from unseen later seasons.


In [4]:
from sklearn.metrics import accuracy_score

chrono_acc = accuracy_score(y1_test, y_pred_xgb_d1) * 100
comp = pd.DataFrame([
    {'Split': 'Random 80/20',  'XGBoost Tuned (D1) accuracy %': 52.78},
    {'Split': 'Chronological', 'XGBoost Tuned (D1) accuracy %': round(chrono_acc, 2)},
])
print(comp.to_string(index=False))

        Split  XGBoost Tuned (D1) accuracy %
 Random 80/20                          52.78
Chronological                          52.85


## Save the deployed Cycle 1 model

This notebook produces the **deployed** Cycle 1 model — chronological tuning is the honest evaluation, so its winner is what the API serves.


In [5]:
import joblib
from sklearn.preprocessing import StandardScaler

# Refit a clean scaler on the chronological training set
scaler1 = StandardScaler().fit(X1_train)

best_xgb = search_d1.best_estimator_

joblib.dump(best_xgb,                str(Paths.C1_MODEL))
joblib.dump(scaler1,                 str(Paths.C1_SCALER))
joblib.dump(list(X1_train.columns),  str(Paths.C1_FEATURES))

print(f'Model saved    -> {Paths.C1_MODEL}')
print(f'Scaler saved   -> {Paths.C1_SCALER}')
print(f'Features saved -> {Paths.C1_FEATURES}')
print(f'\nDeployed feature columns ({len(X1_train.columns)}): {list(X1_train.columns)}')

Model saved    -> /Users/mac/Desktop/diaa/freelance/football project/FootballPredictor/models/cycle1/cycle1_xgb_best.pkl
Scaler saved   -> /Users/mac/Desktop/diaa/freelance/football project/FootballPredictor/models/cycle1/cycle1_scaler.pkl
Features saved -> /Users/mac/Desktop/diaa/freelance/football project/FootballPredictor/models/cycle1/cycle1_feature_cols.pkl

Deployed feature columns (33): ['HomeTeam', 'AwayTeam', 'HTGS', 'ATGS', 'HTGC', 'ATGC', 'HTP', 'ATP', 'HM1', 'HM2', 'HM3', 'HM4', 'HM5', 'AM1', 'AM2', 'AM3', 'AM4', 'AM5', 'MW', 'HTFormPts', 'ATFormPts', 'HTWinStreak3', 'HTWinStreak5', 'HTLossStreak3', 'HTLossStreak5', 'ATWinStreak3', 'ATWinStreak5', 'ATLossStreak3', 'ATLossStreak5', 'HTGD', 'ATGD', 'DiffPts', 'DiffFormPts']
